# 03 · Gold tables with SQL

**What this notebook does:** turns the cleaned tables into small, ready-to-use **gold** tables: one per question
a manager might ask. The AI/BI dashboard and the Genie space read only these tables, so their numbers always agree.

Every query here uses plain SQL that runs on both Databricks (Spark SQL) and DuckDB.

**One thing to remember:** the archive stops on 14 August 2026, so August 2026 is a partial month.
Monthly trends stop at July 2026.

In [1]:
from radar import setup, run_sql
setup()

'Local: tables go to data/complaint_radar.duckdb'

### Complaints per month and product
This counts all complaints, with or without a narrative. It drives the main trend chart.

In [2]:
run_sql("""
    CREATE OR REPLACE TABLE gold_monthly AS
    SELECT month, product, SUM(complaints) AS complaints
    FROM monthly_counts
    WHERE month < '2026-08-01'                         -- August 2026 is only half a month
    GROUP BY month, product
""")
data = run_sql("SELECT month, SUM(complaints) AS complaints FROM gold_monthly GROUP BY month ORDER BY month")
data.tail(6)

,month,complaints
13,2026-02-01,497120.0
14,2026-03-01,614945.0
15,2026-04-01,618293.0
16,2026-05-01,619940.0
17,2026-06-01,620794.0
18,2026-07-01,672036.0


### Each product's share of all complaints

In [3]:
run_sql("""
    CREATE OR REPLACE TABLE gold_products AS
    SELECT product,
           SUM(complaints) AS complaints,
           ROUND(100.0 * SUM(complaints) / (SELECT SUM(complaints) FROM gold_monthly), 2) AS share_pct
    FROM gold_monthly
    GROUP BY product
""")
data = run_sql("SELECT * FROM gold_products ORDER BY complaints DESC")
data

,product,complaints,share_pct
0,Credit reporting,8596387.0,89.29
1,Debt collection,483591.0,5.02
2,Credit card,146848.0,1.53
3,Bank account,136997.0,1.42
4,Money transfer,107442.0,1.12
5,Mortgage,45049.0,0.47
6,Vehicle loan,35790.0,0.37
7,Student loan,34116.0,0.35
8,Personal loan,24080.0,0.25
9,Prepaid card,10113.0,0.11


### Top issues inside each product
`issue` is what the customer picked from the CFPB form, like "Incorrect information on your report".
We keep the top 8 per product, ranked with a window function.

In [4]:
run_sql("""
    CREATE OR REPLACE TABLE gold_issues AS
    WITH counted AS (
        SELECT product, issue, SUM(complaints) AS complaints
        FROM monthly_counts
        WHERE month < '2026-08-01' AND issue IS NOT NULL
        GROUP BY product, issue
    ), ranked AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY product ORDER BY complaints DESC) AS rank_in_product,
               ROUND(100.0 * complaints / SUM(complaints) OVER (PARTITION BY product), 2) AS share_of_product_pct
        FROM counted
    )
    SELECT * FROM ranked WHERE rank_in_product <= 8
""")
data = run_sql("SELECT * FROM gold_issues WHERE product = 'Credit card' ORDER BY rank_in_product")
data

,product,issue,complaints,rank_in_product,share_of_product_pct
0,Credit card,Problem with a purchase shown on your statement,38842.0,1,26.45
1,Credit card,Incorrect information on your report,17630.0,2,12.01
2,Credit card,Getting a credit card,17318.0,3,11.79
3,Credit card,"Other features, terms, or problems",14956.0,4,10.18
4,Credit card,Fees or interest,13761.0,5,9.37
5,Credit card,Problem with a company's investigation into an...,12317.0,6,8.39
6,Credit card,Closing your account,9978.0,7,6.79
7,Credit card,Problem when making payments,7547.0,8,5.14


### How companies respond
For each company with at least 500 complaints:
* **timely %**: share of complaints answered within the CFPB's 15-day window
* **relief %**: share closed with money back or another fix ("monetary" or "non-monetary relief")
* **explanation only %**: share closed with just an explanation

These are complaint patterns, not a verdict on any company: big companies get more complaints simply because
they have more customers.

In [5]:
run_sql("""
    CREATE OR REPLACE TABLE gold_companies AS
    SELECT company,
           SUM(complaints) AS complaints,
           ROUND(100.0 * SUM(CASE WHEN timely = 'Yes' THEN complaints ELSE 0 END) / SUM(complaints), 1) AS timely_pct,
           ROUND(100.0 * SUM(CASE WHEN company_response IN ('Closed with monetary relief', 'Closed with non-monetary relief')
                                  THEN complaints ELSE 0 END) / SUM(complaints), 1) AS relief_pct,
           ROUND(100.0 * SUM(CASE WHEN company_response = 'Closed with explanation' THEN complaints ELSE 0 END) / SUM(complaints), 1) AS explanation_pct,
           ROUND(100.0 * SUM(CASE WHEN company_response = 'In progress' THEN complaints ELSE 0 END) / SUM(complaints), 1) AS in_progress_pct
    FROM company_counts
    GROUP BY company
    HAVING SUM(complaints) >= 500
""")
data = run_sql("SELECT * FROM gold_companies ORDER BY complaints DESC LIMIT 15")
data

,company,complaints,timely_pct,relief_pct,explanation_pct,in_progress_pct
0,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",3189937.0,100.0,50.5,45.4,4.1
1,"EQUIFAX, INC.",3019830.0,100.0,52.8,45.7,1.5
2,Experian Information Solutions Inc.,2722980.0,100.0,3.9,85.7,10.4
3,"Block, Inc.",58914.0,98.4,0.1,99.5,0.4
4,CAPITAL ONE FINANCIAL CORPORATION,58022.0,100.0,15.7,83.1,1.2
5,"CBC Companies, Inc.",47558.0,100.0,58.6,41.3,0.0
6,LEXISNEXIS,44349.0,99.1,26.3,73.5,0.2
7,JPMORGAN CHASE & CO.,42669.0,100.0,14.4,84.8,0.8
8,Resurgent Capital Services L.P.,40054.0,100.0,27.6,72.3,0.1
9,CL Holdings LLC,36922.0,100.0,25.5,74.4,0.1


### Which product each company's complaints are about

In [6]:
run_sql("""
    CREATE OR REPLACE TABLE gold_company_products AS
    SELECT c.company, c.product, SUM(c.complaints) AS complaints
    FROM company_counts c
    JOIN gold_companies g ON g.company = c.company          -- only companies big enough to be in the scorecard
    GROUP BY c.company, c.product
""")
data = run_sql("SELECT COUNT(DISTINCT company) AS companies, COUNT(*) AS rows_ FROM gold_company_products")
data

,companies,rows_
0,249,1813


### How responses differ by product

In [7]:
run_sql("""
    CREATE OR REPLACE TABLE gold_responses AS
    SELECT product, company_response, SUM(complaints) AS complaints,
           ROUND(100.0 * SUM(complaints) / SUM(SUM(complaints)) OVER (PARTITION BY product), 2) AS share_of_product_pct
    FROM company_counts
    WHERE company_response IS NOT NULL
    GROUP BY product, company_response
""")
data = run_sql("SELECT * FROM gold_responses WHERE product = 'Bank account' ORDER BY complaints DESC")
data

,product,company_response,complaints,share_of_product_pct
0,Bank account,Closed with explanation,117284.0,81.14
1,Bank account,Closed with monetary relief,16249.0,11.24
2,Bank account,Closed with non-monetary relief,7051.0,4.88
3,Bank account,In progress,3787.0,2.62
4,Bank account,Untimely response,178.0,0.12


### Fastest-growing issues
Compares the last three full months (May to July 2026) with the same three months in 2025.
Only issues with at least 300 complaints in both periods, so tiny numbers don't look like huge jumps.

In [8]:
run_sql("""
    CREATE OR REPLACE TABLE gold_issue_growth AS
    WITH periods AS (
        SELECT product, issue,
               SUM(CASE WHEN month >= '2026-05-01' AND month < '2026-08-01' THEN complaints ELSE 0 END) AS recent,
               SUM(CASE WHEN month >= '2025-05-01' AND month < '2025-08-01' THEN complaints ELSE 0 END) AS year_ago
        FROM monthly_counts
        WHERE issue IS NOT NULL
        GROUP BY product, issue
    )
    SELECT product, issue, recent, year_ago,
           ROUND(100.0 * (recent - year_ago) / year_ago, 1) AS change_pct
    FROM periods
    WHERE recent >= 300 AND year_ago >= 300
""")
data = run_sql("SELECT * FROM gold_issue_growth ORDER BY change_pct DESC LIMIT 10")
data

,product,issue,recent,year_ago,change_pct
0,Credit reporting,Problem with a company's investigation into an...,2099.0,687.0,205.5
1,Debt collection,Took or threatened to take negative or legal a...,20591.0,7848.0,162.4
2,Personal loan,Problem when making payments,852.0,411.0,107.3
3,Money transfer,"Managing, opening, or closing your mobile wall...",718.0,389.0,84.6
4,Credit reporting,Problem with a company's investigation into an...,351608.0,196090.0,79.3
5,Mortgage,Struggling to pay mortgage,2597.0,1493.0,73.9
6,Money transfer,Unauthorized transactions or other transaction...,1763.0,1063.0,65.9
7,Credit reporting,Problem with fraud alerts or security freezes,5462.0,3374.0,61.9
8,Credit card,Problem with a purchase shown on your statement,7945.0,4933.0,61.1
9,Credit reporting,Incorrect information on your report,1045897.0,668285.0,56.5
